[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/mirror_energy/mirror_energy.ipynb)

# Energy in a Mirror

This example computes the kinetic energy of a body of three masses turning on a sphere, and of its mirror image. The mirror image turns the other way, yet its kinetic energy is the same positive number, whether the mirror is applied to the body's inertia or the inertia is built again from the reflected masses.

On the sphere, in the algebra `x+y+z+`, positions are modelled as intrinsic antivectors and the rate of turning as an extrinsic bivector. A mass's momentum joins its position with its velocity, `point.commutator(rate)`, and is intrinsic; the inertia is a map from rates to momenta, and the energy is half the regressive product `momentum & rate`, an intrinsic element paired with an extrinsic one. Such a pairing needs no repair when the body is changed, and a mirror is one test of that: one reflection, `mirror >> x`, acts on every element, the orientation each element carries settles its sign, and the energy keeps its value. A sandwich that treats every element alike, `mirror * x.involute() * mirror.inverse()`, gives the momentum the opposite sign and the mirrored body a negative energy.

In the vector notation of rigid-body mechanics, the energy reads $T = \tfrac12\,\boldsymbol\omega\cdot\mathsf I\,\boldsymbol\omega$ with the inertia tensor $\mathsf I = \sum_i m_i\,(|\mathbf r_i|^2\,\mathbb 1 - \mathbf r_i\mathbf r_i^{\mathsf T})$, and a mirror $M$ takes each position to $M\mathbf r_i$ but the angular velocity, an axial vector, to $\det(M)\,M\boldsymbol\omega$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import Spherical3D
from examples.util.animation import show_animation
from examples.orientation.mirror_energy import render

np.set_printoptions(precision=3, suppress=True)

In [ ]:
ga = Spherical3D
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Point = ga.gatype.antivector()                # Antivector-
Rate = ga.gatype.bivector()                   # Bivector+

## 1. Kinetic energy of the turning masses

Each mass contributes its mass times `points & points.commutator(Rate)`, its position joined with its velocity at the rate; their sum is the body's inertia, a map from rates to momenta. Half the momentum paired with the rate, `momentum & rate`, is the kinetic energy.

In [ ]:
points = mv.point([[1, 0, 0], [0, 0.8, 0.6], [-0.6, 0, -0.8]])   # [masses] Point-
masses = np.array([1, 1.5, 0.5])
rate = mv.xy                                                     # [] Rate+

In [ ]:
# Inertia maps the body's rate of motion to its momentum.
inertia = (points & points.commutator(Rate) * masses).sum(axis=0)  # [] Momentum- <- Rate+
momentum = inertia(rate)                                         # [] Momentum-
energy = (momentum & rate) / 2                                   # [] Scalar+

print("Energy")
print(energy.coefficient_table())

## 2. The mirrored body

The mirror acts on the inertia on its input and its output, `mirror >> inertia(mirror << Rate)`, and gives the same map as the inertia built from the reflected masses, so both routes give the mirrored body the energy of the original. In the animation, the right-hand body is the reflection of the left-hand one and turns the other way. Colours identify matching masses, ball sizes show the masses, and arrows show their velocities.

In [ ]:
mirror = mv.x                                                    # [] Plane+
n_frames = 36
frame_duration_ms = 90
times = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

In [ ]:
# Reflect the motion, and the inertia map on its input and its output.
reflected_rate = mirror >> rate                                  # [] Rate+
reflected_inertia = mirror >> inertia(mirror << Rate)            # [] Momentum- <- Rate+
reflected_energy = (reflected_inertia(reflected_rate) & reflected_rate) / 2  # [] Scalar+

# Building inertia from the reflected masses gives the same result.
reflected_points = mirror >> points                              # [masses] Point-
rebuilt_inertia = (reflected_points & reflected_points.commutator(Rate) * masses).sum(axis=0)  # [] Momentum- <- Rate+
rebuilt_energy = (rebuilt_inertia(reflected_rate) & reflected_rate) / 2  # [] Scalar+

# The prescribed turn and its mirror; a point turning at a rate moves along its commutator with the rate.
turns = (rate * (-times[:, None] / 2)).exp()                     # [times, 1] Rotor+
positions = turns >> points                                      # [times, masses] Point-
velocities = positions.commutator(rate)                          # [times, masses] Velocity-
paths = stack([positions, mirror >> positions], axis=1)          # [times, cases, masses] Point-
tangents = stack([velocities, mirror >> velocities], axis=1)     # [times, cases, masses] Velocity-

print("Reflected energy")
print(reflected_energy.coefficient_table())
print("Rebuilt energy")
print(rebuilt_energy.coefficient_table())
show_animation(render.animate_motion(paths, tangents, masses), "mirror_energy", frame_duration_ms)

## 3. The sandwich without orientation

The sandwich `mirror * x.involute() * mirror.inverse()` acts on every element alike, whether its orientation is intrinsic or extrinsic. The momentum, joined from positions and velocities, is intrinsic, and the rate is extrinsic; the sandwich gives the rate its reflection `mirror >> rate` but the momentum the opposite of `mirror >> momentum`, and the energy of the mirrored body comes out negative.

In [ ]:
# Applying grade involution uniformly discards the distinction between the two orientations.
sandwich_momentum = mirror * momentum.involute() * mirror.inverse()  # [] Momentum-
sandwich_rate = mirror * rate.involute() * mirror.inverse()      # [] Rate+
sandwich_energy = (sandwich_momentum & sandwich_rate) / 2        # [] Scalar+

print("Energy without the orientation distinction")
print(sandwich_energy.coefficient_table())